### Healthcare Supply Chain Optimization

In [0]:
from pyspark.sql import SparkSession
import random
from datetime import datetime, timedelta

spark = SparkSession.builder.getOrCreate()

In [0]:
hospitals = [f"H{str(i).zfill(3)}" for i in range(1,51)]

departments = [
    "ICU","Emergency","Pharmacy","OT","Laboratory",
    "Radiology","Cardiology","Neurology","Oncology",
    "Orthopedics","NICU","PICU","General Ward",
    "Blood Bank","Trauma","Dialysis"
]

products = [
("P1001","Paracetamol","Medicine"),
("P1002","Insulin","Medicine"),
("P1003","IV Fluid","Medicine"),
("P1004","Surgical Mask","PPE"),
("P1005","N95 Mask","PPE"),
("P1006","Gloves","PPE"),
("P1007","Syringe","Consumables"),
("P1008","Blood Tube","Consumables"),
("P1009","Ventilator Filter","Equipment"),
("P1010","Stent","Equipment")
]

suppliers=[
("S001","MedSupply"),
("S002","HealthCare"),
("S003","Prime Pharma"),
("S004","LifeMed"),
("S005","BioCare")
]

In [0]:
from pyspark.sql.types import *
from pyspark.sql import Row

rows=[]

start=datetime(2025,1,1)

for i in range(500000):

    product=random.choice(products)
    supplier=random.choice(suppliers)

    stock=random.randint(50,3000)

    reorder=random.randint(100,600)

    consumption=random.randint(5,150)

    disease=random.randint(20,100)

    emergency=random.randint(5,70)

    demand=int(consumption*7+disease+emergency)

    risk="Yes" if stock<reorder else "No"

    rows.append(Row(

        Inventory_ID=i+1,

        Date=start+timedelta(days=random.randint(0,730)),

        Hospital_ID=random.choice(hospitals),

        Department=random.choice(departments),

        Product_ID=product[0],

        Product_Name=product[1],

        Category=product[2],

        Supplier_ID=supplier[0],

        Supplier_Name=supplier[1],

        Current_Stock=stock,

        Reorder_Level=reorder,

        Daily_Consumption=consumption,

        Lead_Time=random.randint(2,15),

        Unit_Cost=__builtins__.round(random.uniform(1,500),2),

        Expiry_Days=random.randint(30,720),

        Bed_Occupancy=random.randint(60,100),

        Disease_Index=disease,

        Emergency_Cases=emergency,

        Supplier_Score=random.randint(80,100),

        Purchase_Order=random.randint(100,5000),

        Demand_Next_7_Days=demand,

        Stockout_Risk=risk
    ))

In [0]:
df=spark.createDataFrame(rows)

display(df)

df.printSchema()

In [0]:
df.write.format("delta")\
.mode("overwrite")\
.saveAsTable("bronze_healthcare_supply")

In [0]:
bronze=spark.table("bronze_healthcare_supply")

In [0]:
silver=bronze.dropDuplicates()

In [0]:
silver=silver.fillna({

"Current_Stock":0,

"Supplier_Score":80,

"Expiry_Days":365

})

In [0]:
silver.write.format("delta")\
.mode("overwrite")\
.saveAsTable("silver_healthcare_supply")

In [0]:
silver=spark.table("silver_healthcare_supply")

In [0]:
from pyspark.sql.functions import round, col

silver=silver.withColumn(

"Inventory_Days",

round(col("Current_Stock")/col("Daily_Consumption"),2)

)

In [0]:
from pyspark.sql.functions import when

silver=silver.withColumn(

"Need_Reorder",

when(

col("Current_Stock")<col("Reorder_Level"),

1

).otherwise(0)

)

In [0]:
silver=silver.withColumn(

"High_Demand",

when(

col("Demand_Next_7_Days")>800,

1

).otherwise(0)

)

In [0]:
silver.write.format("delta")\
.mode("overwrite")\
.saveAsTable("feature_healthcare_supply")

In [0]:
df=spark.table("feature_healthcare_supply")

In [0]:
display(

df.groupBy("Category")

.count()

)

In [0]:
display(

df.groupBy("Supplier_Name")

.avg("Supplier_Score")

)

In [0]:
display(

df.groupBy("Department")

.avg("Demand_Next_7_Days")

)

In [0]:
display(

df.groupBy("Stockout_Risk")

.count()

)

### ML Model – Demand Forecasting (Regression)

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler=VectorAssembler(

inputCols=[

"Current_Stock",

"Daily_Consumption",

"Disease_Index",

"Emergency_Cases",

"Supplier_Score"

],

outputCol="features"

)

data=assembler.transform(df)

In [0]:
train,test=data.randomSplit([0.8,0.2],42)

In [0]:
from pyspark.ml.regression import RandomForestRegressor

rf=RandomForestRegressor(

featuresCol="features",

labelCol="Demand_Next_7_Days"

)

model=rf.fit(train)

In [0]:
pred=model.transform(test)

display(pred.select(

"Demand_Next_7_Days",

"prediction"

))

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

RegressionEvaluator(

labelCol="Demand_Next_7_Days",

predictionCol="prediction",

metricName="rmse"

).evaluate(pred)

In [0]:
data=df.withColumn(

"label",

when(col("Stockout_Risk")=="Yes",1).otherwise(0)

)

In [0]:
assembler=VectorAssembler(

inputCols=[

"Current_Stock",

"Reorder_Level",

"Daily_Consumption",

"Disease_Index",

"Supplier_Score"

],

outputCol="features"

)

data=assembler.transform(data)

In [0]:
from pyspark.ml.classification import RandomForestClassifier

train,test=data.randomSplit([0.8,0.2],42)

rf=RandomForestClassifier()

model=rf.fit(train)

pred=model.transform(test)

display(pred.select(

"Stockout_Risk",

"prediction"

))

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

MulticlassClassificationEvaluator(

metricName="accuracy"

).evaluate(pred)

In [0]:
import mlflow

mlflow.set_experiment("/HealthcareSupplyChain")

In [0]:
pred.write.format("delta")\
.mode("overwrite")\
.saveAsTable("gold_stockout_prediction")